# Wordstat pipeline

Ноутбук:
- читает CSV или XLSX;
- нормализует колонки;
- отбирает вероятные статьи;
- собирает ключевые фразы;
- считает частотности в Wordstat;
- сохраняет все результаты в один Excel-файл.

Работа начинается с того, что в `INPUT_PATH` указывается путь к исходному файлу. После этого ячейки запускаются последовательно сверху вниз, и ноутбук шаг за шагом обрабатывает данные.

Сначала он читает исходный файл и подготавливает таблицу к работе. Затем формируется единый Excel-файл, в котором будут храниться ключи, их частотности и текущие статусы обработки. Этот файл становится основным рабочим результатом: именно в него ноутбук записывает все изменения по мере выполнения.

Дальше ноутбук проверяет, какие строки уже были обработаны раньше. Если у ключа уже стоит статус `done`, повторно в Wordstat он не отправляется. Благодаря этому можно в любой момент открыть тот же файл и продолжить работу без потери прогресса.

Если в таблице появляются новые ключи или остаются строки, которые еще не были завершены, ноутбук не создает отдельный новый результат, а дозаписывает их в тот же самый Excel-файл. Так постепенно накапливается один общий итоговый документ, который заполняется по мере обработки.

Во время обращения к Wordstat ноутбук работает до тех пор, пока сервис принимает запросы. Если Wordstat сообщает об ограничении по квоте, ноутбук не теряет уже собранные данные: он сначала сохраняет текущий прогресс в файл, а затем останавливает выполнение. За счет этого обработку можно будет продолжить позже с того места, на котором она прервалась.

### Статусы

- `pending` — ключ еще не обработан;
- `done` — частотность получена;
- `error` — при запросе возникла ошибка.

### Вывод

Вся логика построена так, чтобы работать безопасно и постепенно:
- один итоговый файл постоянно дозаполняется;
- уже готовые ключи не пересчитываются;
- фильтрация отсекает шумные страницы до Wordstat;
- при остановке по квоте прогресс не теряется.

За счет этого ноутбук подходит для больших файлов и многоэтапной обработки в несколько запусков.


## Настройки и правила

Эта ячейка задает общие настройки ноутбука: названия колонок, правила фильтрации страниц и правила сборки поисковых запросов.

In [1]:
from pathlib import Path
from urllib.parse import parse_qsl, urlparse
import os
import joblib
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 160)

CFG = {
    'columns': {
        'url': [
            'Address', 'URL', 'url', 'link', 'href', 'address', 'Адрес',
            'Final URL', 'Final Address', 'Page URL', 'Current URL', 'Source'
        ],
        'title': [
            'Title 1', 'Title', 'title', 'meta_title'
        ],
        'h1': [
            'H1-1', 'H1', 'h1'
        ],
        'meta_description': [
            'Meta Description 1', 'Meta Description', 'meta_description', 'description'
        ],
    },

    'page_filter': {
        'url_noise_fragments': [
            'text/html'
        ],

        'min_title_words': 4,
        'min_meta_len': 40,
        'min_combined_text_len': 80,

        'profile_markers': [
            'profile', 'profiles', 'account', 'accounts', 'cabinet',
            'author', 'authors', 'member', 'members', 'user', 'users'
        ],

        'profile_regex': [
            '^user\\d+$', '^author\\d+$', '^member\\d+$'
        ],

        'trash_path_markers': [
            'tag', 'tags', 'search', 'login', 'signup', 'register', 'feed', 'rss',
            'xmlrpc', 'amp', 'archive', 'archives', 'sitemap', 'category', 'categories'
        ],

        'list_path_markers': [
            'catalog', 'collection', 'collections', 'topic', 'topics', 'rubric',
            'rubrics', 'section', 'sections', 'forum', 'forums', 'thread', 'threads',
            'community', 'communities', 'discussion', 'discussions', 'question',
            'questions', 'faq'
        ],

        'section_root_markers': [
            'articles', 'article', 'knowledge', 'blog', 'blogs', 'news', 'guide',
            'guides', 'journal', 'media', 'stories', 'story', 'posts', 'post',
            'wiki', 'help'
        ],

        'service_path_markers': [
            'form', 'quiz', 'test', 'calculator', 'compare', 'comparison',
            'navigator', 'subscription', 'subscribe', 'tariff', 'pricing', 'price',
            'payment', 'checkout', 'cart', 'order', 'help'
        ],

        'document_extensions': [
            '.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx'
        ],

        'document_path_markers': [
            'cdoc', 'doc', 'docs', 'document', 'documents', 'pdf', 'download',
            'file', 'attachment', 'appendix', 'blank', 'tool'
        ],

        'document_view_markers': [
            'view', 'preview', 'open'
        ],

        'document_title_hints': [
            'письмо', 'приказ', 'постановление', 'определение',
            'федеральный закон', 'кодекс', 'норматив', 'правовой акт'
        ],

        'legal_markers': [
            'оферта', 'договор', 'лиценз', 'правил', 'политик', 'конфиденц',
            'персональн', 'соглас', 'реквизит', 'ваканс', 'подписк', 'тариф', 'оплат',
            'agreement', 'policy', 'privacy', 'license', 'legal', 'cookie',
            'refund', 'delivery', 'returns', 'contact', 'contacts', 'about', 'terms'
        ],

        'list_text_markers': [
            'подборка статей', 'все материалы', 'материалы о',
            'подборка материалов', 'каталог', 'список материалов', 'форум',
            'сообщество', 'обсуждение', 'вопросы и ответы', 'faq',
            'каталог статей', 'catalog', 'forum', 'community'
        ],

        'hub_text_markers': [
            'портал о', 'сайт о', 'все о ', 'все про ', 'энциклопедия', 'медиа',
            'каталог', 'рубрика', 'раздел'
        ],

        'bad_h1_markers': [
            'max-image-preview', 'max-snippet', 'max-video-preview'
        ],

        'review_path_markers': [
            'review', 'reviews', 'otzyv', 'otzyvy', 'feedback', 'comments', 'comment'
        ],

        'review_text_markers': [
            'отзывы', 'отзыв', 'мнение владельцев', 'мнения владельцев',
            'что говорят владельцы', 'комментарии', 'review', 'reviews', 'feedback'
        ],

        'news_path_markers': [
            'news', 'novosti', 'press', 'press-center', 'presscentre', 'pressroom',
            'media', 'events', 'event', 'announcements', 'announcement', 'updates',
            'release', 'releases'
        ],

        'entertainment_path_markers': [
            'games', 'fun', 'memes', 'meme', 'video', 'videos', 'movie', 'movies',
            'serial', 'series', 'shows', 'show', 'celebrity', 'celebrities',
            'stars', 'afisha'
        ],

        'news_text_markers': [
            'сегодня', 'вчера', 'стало известно', 'сообщили', 'объявили',
            'представили', 'пресс-служба', 'по данным', 'рассказали', 'заявили',
            'опубликовали', 'breaking', 'latest news', 'news', 'новости', 'анонс',
            'анонсировали'
        ],

        'entertainment_text_markers': [
            'афиша', 'премьера', 'юмор', 'мемы', 'мем', 'развлечения', 'тест'
        ],

        'entertainment_text_regex': [
            '\\bтест\\s*:\\s*ка(кой|кая|кое|кие)\\b',
            '\\bка(кой|кая|кое|кие)\\s+ты\\b',
            '\\bинтересн(ые|ых|ый|ая)\\s+факт(ы|а|ов)?\\b',
            '\\bлюбопытн(ые|ых|ый|ая)\\s+факт(ы|а|ов)?\\b',
            '\\bудивительн(ые|ых|ый|ая)\\s+факт(ы|а|ов)?\\b'
        ],

        'dated_text_regex': [
            '\\b20(1\\d|2\\d|3\\d)\\b',
            '\\b\\d{1,2}\\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\\b',
            '\\b(январь|февраль|март|апрель|май|июнь|июль|август|сентябрь|октябрь|ноябрь|декабрь)\\s+20(1\\d|2\\d|3\\d)\\b'
        ],

        'pagination_regex': [
            '[?&]page=\\d+',
            '/page/\\d+/?$',
            '/p/\\d+/?$'
        ],

        'sort_filter_params': [
            'page', 'paged', 'sort', 'order', 'filter', 'filters', 'q', 'query', 'search'
        ],

        'media_last_segments': [
            'foto', 'photo', 'photos', 'video', 'videos',
            'gallery', 'galereya', 'images', 'image', 'img', 'amp', 'print'
        ],

        'media_path_markers': [
            '/foto/', '/photo/', '/video/', '/gallery/', '/images/', '/amp/', '/print/'
        ],

        'non_article_first_segments': [
            'program', 'programs', 'course', 'courses', 'trainer', 'trainers',
            'coach', 'coaches', 'page', 'pages', 'profile', 'profiles',
            'author', 'authors', 'brand', 'brands', 'shop', 'product', 'products'
        ],

        'info_url_markers': [
            'knowledge', 'blog', 'article', 'articles', 'guide', 'guides', 'wiki',
            'care', 'health', 'prevention', 'nutrition', 'feeding', 'training',
            'behavior', 'behaviour', 'adoption', 'breed', 'breeds', 'disease', 'diseases', 'faq', 'review',
            'howto', 'tips'
        ],

        'non_info_url_markers': [
            'news', 'new', 'allnews', 'media', 'event', 'events', 'afisha',
            'promo', 'action', 'actions', 'sale', 'sales', 'contest', 'draw',
            'festival', 'fun', 'video', 'videos', 'photo', 'photos', 'foto',
            'gallery', 'stories', 'story', 'interview', 'program', 'programs',
            'course', 'courses', 'trainer', 'trainers', 'coach', 'coaches',
            'page', 'pages', 'brand', 'brands', 'shop', 'product', 'products',
            'catalog'
        ],

        'non_info_text_markers': [
            'новость', 'новости', 'сегодня', 'теперь', 'появился', 'появилась',
            'появились', 'пройдет', 'прошел', 'выдали', 'запустили', 'добавил',
            'добавила', 'видео', 'фото', 'ролик', 'соцсет', 'вирусн', 'мем',
            'шоу', 'конкурс', 'розыгрыш', 'акция', 'скидка', 'распродажа',
            'фестиваль', 'интервью'
        ],

        'common_first_names': [
            'александр', 'саша', 'шура',
            'алексей', 'леша', 'лёша',
            'андрей', 'андрюша',
            'антон',
            'артем', 'артём', 'тема', 'тёма',
            'аркадий', 'аркаша',
            'борис', 'боря',
            'вадим',
            'валентин', 'валя',
            'валерий', 'лера',
            'василий', 'вася',
            'виктор', 'витя',
            'виталий', 'виталий', 'витя',
            'владимир', 'вова',
            'владислав', 'влад', 'слава',
            'вячеслав', 'слава',
            'григорий', 'гриша',
            'геннадий', 'гена',
            'георгий', 'гоша', 'жора',
            'денис',
            'дмитрий', 'дима',
            'даниил', 'данил', 'данила', 'даня',
            'егор',
            'евгений', 'женя',
            'иван', 'ваня',
            'игорь',
            'илья',
            'константин', 'костя',
            'кирилл', 'кира',
            'леонид', 'леня', 'лёня',
            'максим', 'макс',
            'михаил', 'миша',
            'матвей', 'мотя',
            'марк',
            'никита',
            'николай', 'коля',
            'олег',
            'павел', 'паша',
            'петр', 'пётр', 'петя',
            'роман', 'рома',
            'ростислав', 'ростик',
            'руслан',
            'сергей', 'сережа', 'серёжа',
            'станислав', 'стас',
            'семен', 'семён', 'сеня',
            'тимур',
            'федор', 'фёдор', 'федя', 'федор', 'фёдор',
            'юрий', 'юра',
            'ярослав', 'ярик',

            'анна', 'аня',
            'анастасия', 'настя',
            'антонина', 'тоня',
            'алена', 'алёна', 'лена', 'лёна',
            'алиса',
            'арина',
            'валерия', 'лера',
            'валентина', 'валя',
            'виктория', 'вика',
            'вера',
            'вероника', 'ника',
            'галина', 'галя',
            'дарья', 'дария', 'даша',
            'диана',
            'евгения', 'женя',
            'екатерина', 'катя',
            'елена', 'лена',
            'елизавета', 'лиза',
            'зоя',
            'ирина', 'ира',
            'карина',
            'ксения', 'ксюша',
            'лариса', 'лара',
            'лилия', 'лиля', 'лиза',
            'лидия', 'лида',
            'любовь', 'люба',
            'мария', 'маша',
            'марина',
            'маргарита', 'рита',
            'милана',
            'мирослава', 'мира',
            'наталья', 'наташа',
            'надежда', 'надя',
            'оксана',
            'ольга', 'оля',
            'полина', 'поля',
            'светлана', 'света',
            'софия', 'софья', 'соня',
            'таисия', 'тая',
            'тамара', 'тома',
            'татьяна', 'таня',
            'юлия', 'юля',
            'яна'
        ],

        'first_person_markers': [
            'я', 'мы', 'мой', 'моя', 'мои', 'мое', 'моё', 'наш', 'наша', 'наши', 'наше'
        ],

        'first_person_starts': [
            'как я ', 'как мы ', 'почему я ', 'почему мы ', 'зачем я ', 'зачем мы '
        ],
    },

    'query_builder': {
        'tail_noise_patterns': [
            '\\bс фото\\b.*$',
            '\\bс видео\\b.*$',
            '\\bи видео\\b.*$',
            '\\bфото\\b.*$',
            '\\bвидео\\b.*$',
            '\\bдля начинающих\\b.*$',
            '\\bпошагов[а-я]*\\b.*$',
            '\\b\\d{4}\\b.*$'
        ],
        'split_regex': '\\s*[:|]\\s*|\\s+—\\s+|\\s+\\-\\s+',
        'max_query_tokens': 7
    }
}

## Загрузка и нормализация таблицы

Эта ячейка загружает CSV или Excel, очищает названия колонок и приводит таблицу к единому формату.

In [2]:
# Нормализуем название колонки: убираем мусор, приводим к нижнему регистру,
# заменяем дефисы и подчеркивания на пробелы.
def normalize_colname(name: str) -> str:
    name = str(name).replace('\ufeff', '').strip().lower()
    name = name.strip('"\'`“”«»')
    name = name.replace('_', ' ').replace('-', ' ')
    name = re.sub(r'\s+', ' ', name)
    return name


# Чистим и нормализуем названия колонок во всем датафрейме.
def cleanup_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_colname(col) for col in df.columns]
    return df


# Удаляем колонки, которые целиком пустые.
def drop_fully_empty_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols_to_drop = []

    for col in df.columns:
        values = df[col].fillna('').astype(str).str.strip()
        if (values == '').all():
            cols_to_drop.append(col)

    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    return df


# Удаляем служебные колонки Excel вида "Unnamed: 0", "Unnamed: 1" и т.д.
def drop_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols_to_drop = [col for col in df.columns if str(col).startswith('unnamed:')]

    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    return df


# Читаем CSV или Excel и приводим названия колонок к аккуратному виду.
def load_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in ['.xlsx', '.xls']:
        df = pd.read_excel(path)

    elif suffix == '.csv':
        last_error = None

        for enc in ['utf-8', 'utf-8-sig', 'cp1251', 'windows-1251', 'latin-1']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=None, engine='python')
                break
            except Exception as e:
                last_error = e
        else:
            raise last_error

    else:
        raise ValueError(f'Неподдерживаемый формат: {suffix}')

    df = cleanup_dataframe_columns(df)
    df = drop_fully_empty_columns(df)
    df = drop_unnamed_columns(df)
    return df


# Ищем первую реально существующую колонку из списка возможных названий.
# Это упрощенный и безопасный аналог функции вроде find_first_existing_column.
def find_column_by_aliases(df: pd.DataFrame, aliases: list[str]) -> str | None:
    normalized_aliases = [normalize_colname(alias) for alias in aliases]

    for alias in normalized_aliases:
        if alias in df.columns:
            return alias

    return None


# Приводим таблицу к стандартным колонкам: url, title, h1, meta_description.
# Здесь больше нет повторной очистки колонок: load_table уже сделал ее раньше.
def normalize_input_table(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    columns_cfg = cfg.get('columns', {})
    standard_columns = ['url', 'title', 'h1', 'meta_description']

    found_columns = {
        col: find_column_by_aliases(df, columns_cfg.get(col, []))
        for col in standard_columns
    }

    missing = [col for col, found in found_columns.items() if found is None]
    if missing:
        raise ValueError(
            f'Не удалось найти обязательные колонки: {missing}. '
            f'Найдены колонки: {list(df.columns)}'
        )

    out = pd.DataFrame({
        col: df[found_columns[col]]
        for col in standard_columns
    })

    return out


# Извлекаем основной домен для имени итогового файла.
def extract_main_domain(urls: pd.Series) -> str:
    vals = urls.dropna().astype(str).str.strip()
    vals = vals[vals != '']

    if len(vals) == 0:
        return 'site'

    for value in vals:
        try:
            host = urlparse(value).netloc.lower().replace('www.', '')
            if host:
                return re.sub(r'[^a-z0-9]+', '_', host).strip('_') or 'site'
        except Exception:
            continue

    return 'site'


## Фильтрация страниц

Эта ячейка разбирает URL и текст страницы, определяет причину фильтрации и отделяет вероятные статьи от остальных страниц.

### Что делает код по шагам

1. **Базово очищает таблицу**: заполняет пустые поля, чистит URL, нормализует `Title`, `H1`, `Meta Description`.
2. **Удаляет пустые строки**: если у строки нет ни `Title`, ни `H1`, ни `Meta Description`, она не участвует в анализе.
3. **Проверяет URL**: оставляет корректные адреса `http/https`, убирает дубли.
4. **Разбирает URL на части**: путь, сегменты, параметры, глубину вложенности, первый и последний сегмент.
5. **Сравнивает страницу с набором правил**: по URL, заголовкам и мета-описанию.
6. **Присваивает каждой строке причину** в колонку `filter_reason`.
7. **Делит результат на две таблицы**:
   - `content_df` — страницы, которые прошли фильтр;
   - `filtered_df` — страницы, которые были отсеяны, с указанием причины.

### Что означает каждая причина фильтрации

- `bad_url` — строка не похожа на нормальный URL.
- `media_url` — это медиа-страница: фото, видео, галерея, amp, print и похожие форматы.
- `non_article_section` — страница лежит в явном неинформационном разделе сайта.
- `listing_params` — у URL есть признаки листинга, пагинации, сортировки, поиска или служебных параметров.
- `hub_page` — страница похожа на раздел, хаб или рубрику, а не на отдельную статью.
- `profile_regex` — первый сегмент URL совпал с шаблоном профиля или карточки пользователя.
- `profile_marker` — в URL есть явные маркеры профиля, автора, аккаунта.
- `document_like` — страница похожа на документ: PDF, DOC, XLS, PPT и похожие материалы.
- `trash_path` — URL содержит мусорные или технические пути, которые не относятся к статьям.
- `service_path` — страница относится к служебным разделам: логин, регистрация, feed, rss, sitemap и т. п.
- `bad_h1_marker` — в `H1` есть маркер, который заранее считается плохим для статьи.
- `review_page` — страница похожа на отзывы, рейтинг, обзор товара или пользовательские мнения.
- `news_page` — страница выглядит как новость, а не как evergreen-материал или статья под ключ.
- `entertainment_page` — развлекательный контент: тесты, мемы, шоу, афиша, знаменитости и т. п.
- `dated_page` — в тексте есть сильный признак временной привязки, из-за чего страница похожа на материал под дату.
- `non_info_url` — URL указывает на формат, который не похож на полезную информационную статью.
- `non_info_text` — текст страницы по формулировкам не похож на статью для подбора ключа.
- `legal_or_service` — юридическая, сервисная или policy-страница.
- `first_person_page` — в тексте много сигналов от первого лица: личный блог, история, отзыв, дневниковый формат.
- `person_name_page` — заголовок похож на ФИО или страницу, посвященную конкретному человеку.
- `list_text` — по самому тексту это скорее список, рубрикатор или подборка ссылок, а не полноценная статья.
- `list_path` — по URL это больше похоже на список или раздел, а не на отдельный материал.
- `site_hub_text` — в тексте есть признаки общей страницы сайта, каталога или витрины.
- `too_little_text` — слишком мало текста в `Title`, `H1` и `Meta Description`, чтобы уверенно считать страницу статьей.
- `ok` — страница прошла фильтр и считается кандидатом на дальнейшую обработку.

### Зачем это нужно

Такой фильтр уменьшает шум еще до извлечения ключевых фраз. В Wordstat уходят не все URL подряд, а только те страницы, которые больше похожи на полноценные информационные материалы. Это снижает число пустых или нерелевантных запросов и делает итоговый файл чище.

In [3]:
# Чистим URL от служебного мусора из выгрузки.
def normalize_url(url: str, cfg: dict) -> str:
    if url is None:
        return ''

    text = str(url).strip()

    for noise in cfg.get('page_filter', {}).get('url_noise_fragments', []):
        text = text.replace(str(noise), '')

    text = text.strip(' "\'')
    text = re.sub(r'[,\s]+$', '', text)
    return text


# Проверяем, что строка похожа на обычный URL.
def is_valid_url(text: str) -> bool:
    text = str(text).strip().lower()
    return text.startswith('http://') or text.startswith('https://')

# Базово очищаем текст
def normalize_text(text: str) -> str:
    text = str(text).replace('ё', 'е')
    text = re.sub('<[^>]+>', ' ', text)
    text = text.replace('—', ' - ').replace('–', ' - ')
    text = re.sub(r'[«»"\'`“”]+', ' ', text)
    text = re.sub(r'[\t\r\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_and_tokenize(text: str) -> list[str]:
    text = normalize_text(text).lower()
    return re.findall(r'[a-zа-я0-9]+', text)

# Базовая очистка перед фильтрацией.
def basic_clean(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    work_df = df.copy()

    for col in ['url', 'title', 'h1', 'meta_description']:
        work_df[col] = work_df[col].fillna('').astype(str).str.strip()

    # URL чистим отдельно своей логикой
    work_df['url'] = work_df['url'].apply(lambda x: normalize_url(x, cfg))

    # Текстовые поля нормализуем ДО фильтрации
    for col in ['title', 'h1', 'meta_description']:
        work_df[col] = work_df[col].apply(normalize_text)

    has_any_text = (
        (work_df['title'] != '') |
        (work_df['h1'] != '') |
        (work_df['meta_description'] != '')
    )
    work_df = work_df[has_any_text].copy()

    valid_mask = work_df['url'].apply(is_valid_url)

    if valid_mask.any():
        work_df = work_df[valid_mask].copy()
        work_df = work_df.drop_duplicates(subset=['url'], keep='first')
    else:
        work_df = work_df.drop_duplicates(
            subset=['title', 'h1', 'meta_description'],
            keep='first'
        )

    return work_df.reset_index(drop=True)


# Разбираем URL один раз и возвращаем все нужные части.
def parse_url_parts(url: str) -> dict:
    try:
        parsed = urlparse(str(url).strip())
        path = parsed.path.strip('/').lower()
        segments = [segment for segment in path.split('/') if segment]
        query_keys = [
            str(key).lower().strip()
            for key, _ in parse_qsl(parsed.query, keep_blank_values=True)
        ]

        return {
            'path': path,
            'segments': segments,
            'query_keys': query_keys,
            'depth': len(segments),
            'first_segment': segments[0] if segments else '',
            'last_segment': segments[-1] if segments else '',
            'domain': parsed.netloc.lower().replace('www.', '')
        }
    except Exception:
        return {
            'path': '',
            'segments': [],
            'query_keys': [],
            'depth': 0,
            'first_segment': '',
            'last_segment': '',
            'domain': ''
        }


# Ищем обычное частичное вхождение маркеров в тексте.
def contains_any(text: str, markers: list[str] | set[str]) -> bool:
    text = str(text).lower()
    return any(str(marker).lower() in text for marker in markers)


# Проверяем полное совпадение строки с regex-шаблоном.
def matches_any_regex(text: str, patterns: list[str]) -> bool:
    text = str(text).strip().lower()
    return any(re.fullmatch(pattern, text) for pattern in patterns)


# Ищем regex-шаблон внутри строки.
def matches_any_text_regex(text: str, patterns: list[str]) -> bool:
    text = str(text).lower()
    return any(re.search(pattern, text) for pattern in patterns)


def contains_marker_as_phrase(text: str, markers: list[str] | set[str]) -> bool:
    text = normalize_text(text).lower()
    text = re.sub(r'[_/]+', ' ', text)
    text = re.sub(r'[-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    for marker in markers:
        marker_norm = normalize_text(marker).lower()
        marker_norm = re.sub(r'[_/]+', ' ', marker_norm)
        marker_norm = re.sub(r'[-]+', ' ', marker_norm)
        marker_norm = re.sub(r'\s+', ' ', marker_norm).strip()

        if not marker_norm:
            continue

        pattern = rf'(?<!\w){re.escape(marker_norm)}(?!\w)'
        if re.search(pattern, text):
            return True

    return False


# Проверяем сегмент URL на безопасное совпадение с маркером.
# Срабатывает только на полное совпадение сегмента или его частей после split по "-" и "_".
def segment_matches_marker(segment: str, marker: str) -> bool:
    segment = str(segment).strip().lower()
    marker = str(marker).strip().lower()

    if not segment or not marker:
        return False

    if segment == marker:
        return True

    parts = [part for part in re.split(r'[-_]', segment) if part]
    return marker in parts


# Ищем маркеры в сегментах URL без грубого частичного вхождения.
def path_matches_markers(url_parts: dict, markers: list[str]) -> bool:
    segments = [
        str(segment).strip().lower()
        for segment in url_parts.get('segments', [])
        if str(segment).strip()
    ]
    normalized_markers = [
        str(marker).strip().lower()
        for marker in markers
        if str(marker).strip()
    ]

    for segment in segments:
        for marker in normalized_markers:
            if segment_matches_marker(segment, marker):
                return True

    return False


# Есть ли признаки пагинации, сортировки, фильтрации или поиска.
def has_pagination_or_sort_signals(url: str, url_parts: dict, pf: dict) -> bool:
    low_url = str(url).lower()

    for pattern in pf.get('pagination_regex', []):
        try:
            if re.search(pattern, low_url):
                return True
        except re.error:
            continue

    blocked_params = {
        str(value).lower()
        for value in pf.get('sort_filter_params', [])
    }
    query_keys = url_parts.get('query_keys', [])

    return any(key in blocked_params for key in query_keys)


# Это страница-хаб, раздел или главная.
def is_hub_like(url_parts: dict, title: str, h1: str, meta: str, pf: dict) -> bool:
    path = url_parts.get('path', '')
    segments = url_parts.get('segments', [])
    combined = f'{title} {h1} {meta}'.strip().lower()

    if path == '':
        return True

    section_roots = {
        str(value).lower()
        for value in pf.get('section_root_markers', [])
    }
    if len(segments) == 1 and segments[0] in section_roots:
        return True

    hub_text_markers = [str(value).lower() for value in pf.get('hub_text_markers', [])]
    return len(segments) <= 2 and any(marker in combined for marker in hub_text_markers)


# Это документ или файловая страница.
def is_document_like(url_parts: dict, title: str, pf: dict) -> bool:
    path = url_parts.get('path', '')
    segments = url_parts.get('segments', [])
    low_title = str(title).lower()

    document_extensions = tuple(
        str(value).lower()
        for value in pf.get('document_extensions', [])
    )
    if document_extensions and path.endswith(document_extensions):
        return True

    document_path_markers = {
        str(value).lower()
        for value in pf.get('document_path_markers', [])
    }
    if any(seg in document_path_markers for seg in segments):
        return True

    document_view_markers = {
        str(value).lower()
        for value in pf.get('document_view_markers', [])
    }
    if any(seg in document_view_markers for seg in segments) and any(seg in document_path_markers for seg in segments):
        return True

    document_title_hints = [
        str(value).lower()
        for value in pf.get('document_title_hints', [])
    ]
    if any(hint in low_title for hint in document_title_hints):
        return True

    return False


# Это медиа-страница.
def is_media_like_url(url_parts: dict, pf: dict) -> bool:
    path = url_parts.get('path', '')
    last_segment = url_parts.get('last_segment', '')

    media_last_segments = {
        str(value).lower()
        for value in pf.get('media_last_segments', [])
    }
    if last_segment in media_last_segments:
        return True

    low_path = f"/{str(path).strip('/').lower()}/"
    media_path_markers = [
        str(value).lower()
        for value in pf.get('media_path_markers', [])
    ]
    return any(marker in low_path for marker in media_path_markers)


# Есть ли признаки первого лица.
def has_first_person_signal(text: str, pf: dict) -> bool:
    low = str(text).lower().strip()

    starts = tuple(pf.get('first_person_starts', []))
    if starts and low.startswith(starts):
        return True

    tokens = re.findall(r'[a-zа-я0-9]+', low)
    first_person_markers = set(pf.get('first_person_markers', []))
    return bool(set(tokens) & first_person_markers)


# Похоже ли название страницы на имя человека.
# Проверка стала строже: без угадывания по обрезанным основам слов.
def is_person_name_text(text: str, pf: dict) -> bool:
    tokens = normalize_and_tokenize(text)
    if not tokens:
        return False

    first_names = {
        normalize_text(x).lower()
        for x in pf.get('common_first_names', [])
        if str(x).strip()
    }

    if not first_names:
        return False

    first_tokens = tokens[:3]

    if first_tokens and first_tokens[0] in first_names:
        return True

    if len(first_tokens) >= 2 and first_tokens[0] in first_names:
        second = first_tokens[1]
        if re.fullmatch(r'[a-zа-я]{3,}', second):
            return True

    return False


# Возвращаем причину, по которой страница оставлена или отфильтрована.
def get_filter_reason(row: pd.Series, pf: dict) -> str:
    url = str(row.get('url', '')).strip()
    title = str(row.get('title', '')).strip().lower()
    h1 = str(row.get('h1', '')).strip().lower()
    meta = str(row.get('meta_description', '')).strip().lower()
    combined = f'{title} {h1} {meta}'.strip()

    if not is_valid_url(url):
        return 'bad_url'

    url_parts = parse_url_parts(url)
    path = url_parts.get('path', '')
    first_seg = url_parts.get('first_segment', '')
    last_seg = url_parts.get('last_segment', '')

    title_words = len(title.split()) if title else 0
    h1_words = len(h1.split()) if h1 else 0
    meta_len = len(meta)
    combined_len = len(combined)

    is_list_like_path = (
        path_matches_markers(url_parts, pf.get('list_path_markers', [])) or
        contains_any(first_seg, pf.get('list_path_markers', [])) or
        contains_any(last_seg, pf.get('list_path_markers', []))
    )

    is_list_like_text = (
        contains_any(title, pf.get('list_text_markers', [])) or
        contains_any(h1, pf.get('list_text_markers', [])) or
        contains_any(meta, pf.get('list_text_markers', []))
    )

    if is_media_like_url(url_parts, pf):
        return 'media_url'

    non_article_first_segments = {
        str(value).lower()
        for value in pf.get('non_article_first_segments', [])
    }
    if first_seg in non_article_first_segments:
        return 'non_article_section'

    if has_pagination_or_sort_signals(url, url_parts, pf):
        return 'listing_params'

    if is_hub_like(url_parts, title, h1, meta, pf):
        return 'hub_page'

    if matches_any_regex(first_seg, pf.get('profile_regex', [])):
        return 'profile_regex'

    if contains_any(first_seg, pf.get('profile_markers', [])) or contains_any(path, pf.get('profile_markers', [])):
        return 'profile_marker'

    if is_document_like(url_parts, title, pf):
        return 'document_like'

    if path_matches_markers(url_parts, pf.get('trash_path_markers', [])):
        return 'trash_path'

    if path_matches_markers(url_parts, pf.get('service_path_markers', [])):
        return 'service_path'

    if contains_any(h1, pf.get('bad_h1_markers', [])):
        return 'bad_h1_marker'

    if path_matches_markers(url_parts, pf.get('review_path_markers', [])) or contains_any(combined, pf.get('review_text_markers', [])):
        return 'review_page'

    if path_matches_markers(url_parts, pf.get('news_path_markers', [])) or contains_any(combined, pf.get('news_text_markers', [])):
        return 'news_page'

    if (
        path_matches_markers(url_parts, pf.get('entertainment_path_markers', [])) or
        contains_any(combined, pf.get('entertainment_text_markers', [])) or
        matches_any_text_regex(combined, pf.get('entertainment_text_regex', []))
    ):
        return 'entertainment_page'

    if matches_any_text_regex(combined, pf.get('dated_text_regex', [])):
        return 'dated_page'

    if path_matches_markers(url_parts, pf.get('non_info_url_markers', [])):
        return 'non_info_url'

    if contains_any(combined, pf.get('non_info_text_markers', [])):
        return 'non_info_text'

    if (
        path_matches_markers(url_parts, pf.get('legal_markers', [])) or
        contains_marker_as_phrase(combined, pf.get('legal_markers', []))
    ):
        return 'legal_or_service'

    if has_first_person_signal(combined, pf):
        return 'first_person_page'

    if is_person_name_text(h1 or title, pf):
        return 'person_name_page'

    if is_list_like_text:
        return 'list_text'

    if is_list_like_path and (title_words <= 3 or h1_words <= 3 or meta_len < pf.get('min_meta_len', 40)):
        return 'list_path'

    if contains_any(combined, pf.get('hub_text_markers', [])):
        return 'site_hub_text'

    if (
        title_words < pf.get('min_title_words', 4) and
        h1_words < pf.get('min_title_words', 4) and
        meta_len < pf.get('min_meta_len', 40)
    ):
        return 'too_little_text'

    if combined_len < pf.get('min_combined_text_len', 80):
        return 'too_little_text'

    return 'ok'


# Делим страницы на подходящие и отфильтрованные.
def filter_content_pages(df: pd.DataFrame, cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    work_df = basic_clean(df, cfg).copy()
    pf = cfg.get('page_filter', {})

    work_df['filter_reason'] = work_df.apply(
        lambda row: get_filter_reason(row, pf),
        axis=1
    )

    content_df = work_df[work_df['filter_reason'] == 'ok'].copy().reset_index(drop=True)
    filtered_df = work_df[work_df['filter_reason'] != 'ok'].copy().reset_index(drop=True)

    return content_df, filtered_df

## Модель классификации страниц

На этом этапе к правилам добавляется обученная текстовая модель.  
Она не заменяет правила фильтрации, а работает параллельно с ними: для каждой страницы рассчитывается вероятность того, что страница является статьей, после чего можно сравнить решения правил и модели.


In [4]:
# Готовим текстовые признаки для модели и сопоставляем решение правил с решением классификатора.
def normalize_url_for_model_text(url: str) -> str:
    url = str(url).strip()
    if not url:
        return ''

    url = url.lower()
    url = url.replace('"text/html', '').replace("'text/html", '').replace('text/html', '')
    url = url.strip(' "\'')
    url = re.sub(r'[,#?&\s]+$', '', url)

    try:
        parsed = urlparse(url)
        domain = parsed.netloc.replace('www.', '')
        path = parsed.path
        path = re.sub(r'/text/html/?$', '', path)
        path = re.sub(r'/index\.html?$', '', path)
        text = f'{domain} {path}'
    except Exception:
        text = url

    text = text.replace('/', ' ').replace('-', ' ').replace('_', ' ')
    text = re.sub(r'\.html?$', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_model_text(df: pd.DataFrame) -> pd.DataFrame:
    work_df = df.copy()

    for col in ['url', 'title', 'h1', 'meta_description']:
        if col not in work_df.columns:
            work_df[col] = ''
        work_df[col] = work_df[col].fillna('').astype(str).str.strip()

    work_df['url_text'] = work_df['url'].map(normalize_url_for_model_text)
    work_df['text_all'] = (
        work_df['url_text'] + ' [SEP] ' +
        work_df['title'] + ' [SEP] ' +
        work_df['h1'] + ' [SEP] ' +
        work_df['meta_description']
    ).str.replace(r'\s+', ' ', regex=True).str.strip()

    work_df['text_all_len'] = work_df['text_all'].astype(str).str.len()
    return work_df


def build_rule_markup(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    work_df = basic_clean(df, cfg).copy()
    pf = cfg.get('page_filter', {})

    work_df['filter_reason'] = work_df.apply(
        lambda row: get_filter_reason(row, pf),
        axis=1
    )
    work_df['rule_is_article'] = work_df['filter_reason'].eq('ok').astype(int)
    return work_df


def load_article_model_artifacts(cfg: dict) -> tuple | None:
    model_cfg = cfg.get('model_selector', {})
    model_dir = Path(model_cfg.get('artifacts_dir', 'model_artifacts'))

    model_path = model_dir / 'article_selector_logreg.joblib'
    vectorizer_path = model_dir / 'article_selector_tfidf.joblib'
    threshold_path = model_dir / 'article_selector_threshold.joblib'

    if not model_path.exists() or not vectorizer_path.exists():
        return None

    model = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)

    if threshold_path.exists():
        threshold = float(joblib.load(threshold_path))
    else:
        threshold = float(model_cfg.get('default_threshold', 0.65))

    return model, vectorizer, threshold, model_dir


def apply_article_model(rule_df: pd.DataFrame, cfg: dict) -> tuple[pd.DataFrame, dict]:
    work_df = build_model_text(rule_df).copy()
    loaded = load_article_model_artifacts(cfg)

    if loaded is None:
        work_df['model_proba'] = np.nan
        work_df['model_is_article'] = pd.NA
        work_df['rule_vs_model'] = 'model_artifacts_not_found'
        return work_df, {
            'model_loaded': False,
            'threshold': float(cfg.get('model_selector', {}).get('default_threshold', 0.65)),
            'artifacts_dir': str(Path(cfg.get('model_selector', {}).get('artifacts_dir', 'model_artifacts')).resolve())
        }

    model, vectorizer, threshold, model_dir = loaded

    X_model = vectorizer.transform(work_df['text_all'].fillna('').astype(str))
    work_df['model_proba'] = model.predict_proba(X_model)[:, 1]
    work_df['model_is_article'] = (work_df['model_proba'] >= threshold).astype(int)

    work_df['rule_vs_model'] = np.select(
        [
            (work_df['rule_is_article'] == 1) & (work_df['model_is_article'] == 1),
            (work_df['rule_is_article'] == 0) & (work_df['model_is_article'] == 0),
            (work_df['rule_is_article'] == 1) & (work_df['model_is_article'] == 0),
            (work_df['rule_is_article'] == 0) & (work_df['model_is_article'] == 1),
        ],
        [
            'оба считают статьей',
            'оба считают не статьей',
            'правила да, модель нет',
            'правила нет, модель да',
        ],
        default='не определено'
    )

    return work_df, {
        'model_loaded': True,
        'threshold': threshold,
        'artifacts_dir': str(model_dir.resolve())
    }


def make_rule_model_summary(scored_df: pd.DataFrame) -> pd.DataFrame:
    summary_df = (
        scored_df['rule_vs_model']
        .fillna('не определено')
        .value_counts(dropna=False)
        .rename_axis('rule_vs_model')
        .reset_index(name='count')
    )

    if len(scored_df) > 0:
        summary_df['share'] = (summary_df['count'] / len(scored_df)).round(4)
    else:
        summary_df['share'] = 0.0

    return summary_df

## Очистка текста и сборка запросов

Эта ячейка очищает заголовки, отсекает развлекательные и персональные формулировки и собирает итоговые запросы из `H1` или `Title`.

In [5]:
# Извлекаем query из H1, а если H1 пустой — из Title.
def extract_site_tokens(url: str) -> list[str]:
    url = str(url).strip()

    if not url:
        return []

    try:
        host = urlparse(url).netloc.lower().replace('www.', '')
    except Exception:
        return []

    if not host:
        return []

    host = host.split(':')[0]
    host_parts = host.split('.')

    if len(host_parts) >= 2:
        core_host = '.'.join(host_parts[:-1])
    else:
        core_host = host

    tokens = re.findall(r'[a-zа-я0-9]+', core_host)
    stop_tokens = {'www', 'm', 'amp', 'blog', 'news'}

    return [token for token in tokens if len(token) >= 3 and token not in stop_tokens]


def strip_site_prefix(query: str, url: str) -> str:
    query = str(query).strip().lower()
    site_tokens = extract_site_tokens(url)

    if not query or not site_tokens:
        return query

    tokens = query.split()

    while tokens and tokens[0] in site_tokens:
        tokens = tokens[1:]

    return ' '.join(tokens).strip()


def strip_author_tail(query: str) -> str:
    query = str(query).strip()

    patterns = [
        r'\s+от\s+[а-яa-z][а-яa-z\-]+(?:\s+[а-яa-z][а-яa-z\-]+){0,2}$',
        r'\s+с\s+[а-яa-z][а-яa-z\-]+(?:\s+[а-яa-z][а-яa-z\-]+){0,2}$'
    ]

    for pattern in patterns:
        query = re.sub(pattern, '', query, flags=re.IGNORECASE).strip()

    return query


def extract_query_from_page(title: str, h1: str, url: str, cfg: dict) -> tuple[str, str, str]:
    h1 = str(h1).strip()
    title = str(title).strip()

    if h1:
        source_text = h1
        source = 'h1'
    elif title:
        source_text = title
        source = 'title'
    else:
        return '', '', ''

    text = normalize_text(source_text)
    text = re.sub(r'[\(\)\[\]\{\}]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if '?' in text:
        before_q, after_q = text.split('?', 1)
        after_q = after_q.strip(' .,:;!—-|').strip()
        before_q = before_q.strip(' .,:;!—-|').strip()
        text = after_q if after_q else before_q

    if '.' in text:
        text = text.split('.', 1)[0].strip(' .,:;!—-|').strip()

    split_regex = cfg['query_builder']['split_regex']
    parts = re.split(split_regex, text)
    first_part = parts[0].strip(' -–—:|').strip() if parts else text

    query = first_part.lower()
    query = re.sub(r'[^a-zа-я0-9\s\-]', ' ', query)
    query = query.replace('-', ' ')
    query = re.sub(r'\s+', ' ', query).strip()

    query = strip_site_prefix(query, url)
    query = strip_author_tail(query)

    for pattern in cfg['query_builder'].get('tail_noise_patterns', []):
        query = re.sub(pattern, '', query, flags=re.IGNORECASE)

    query = re.sub(r'^топ\s*[-–—]?\s*\d+\s*', '', query, flags=re.IGNORECASE)
    query = re.sub(r'^(?:\d+\s*)+', '', query)
    query = re.sub(r'\s+', ' ', query).strip(' .,:;!—-|').strip()

    max_tokens = cfg['query_builder'].get('max_query_tokens', 7)
    tokens = query.split()
    if len(tokens) > max_tokens:
        query = ' '.join(tokens[:max_tokens])

    if not query:
        return '', '', ''

    return query, source, source_text


# Строим итоговую очередь уникальных запросов уже после фильтрации страниц.
def build_candidate_queries(content_df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    out_cols = ['query', 'url', 'source_field', 'source_text', 'status']

    if content_df is None or len(content_df) == 0:
        return pd.DataFrame(columns=out_cols)

    rows = []

    for row in content_df.itertuples(index=False):
        query, source, source_text = extract_query_from_page(
            getattr(row, 'title', ''),
            getattr(row, 'h1', ''),
            getattr(row, 'url', ''),
            cfg
        )

        if not query:
            continue

        rows.append({
            'query': query,
            'url': normalize_url(getattr(row, 'url', ''), cfg),
            'source_field': source,
            'source_text': source_text,
            'status': 'pending',
        })

    if not rows:
        return pd.DataFrame(columns=out_cols)

    top_keys_df = pd.DataFrame(rows, columns=out_cols)
    top_keys_df['query'] = top_keys_df['query'].fillna('').astype(str).str.strip()
    top_keys_df['url'] = top_keys_df['url'].fillna('').astype(str).str.strip()

    top_keys_df = top_keys_df[
        (top_keys_df['query'] != '') & (top_keys_df['url'] != '')
    ].copy()

    if len(top_keys_df) == 0:
        return pd.DataFrame(columns=out_cols)

    top_keys_df['q_len'] = top_keys_df['query'].str.len()

    top_keys_df = (
        top_keys_df
        .sort_values(['query', 'q_len', 'url'])
        .groupby('query', as_index=False)
        .agg({
            'url': 'first',
            'source_field': 'first',
            'source_text': 'first',
            'status': 'first',
        })
        .sort_values(['query', 'url'], ascending=[True, True])
        .reset_index(drop=True)
    )

    return top_keys_df[out_cols]


## Работа с Wordstat API

Эта ячейка отправляет запросы в Wordstat, сохраняет результаты в один файл и обновляет статусы по ходу обработки.

### Что делает код

1. Берет очередь ключей `queue_df`.
2. По одному отправляет запросы в метод `topRequests`.
3. Для каждого ключа сохраняет:
   - `status_code` — HTTP-код ответа;
   - `total_count` — суммарную частотность;
   - `top_requests` — несколько связанных запросов из ответа API.
4. Каждые `save_every` запросов пересобирает итоговую таблицу и сохраняет ее в тот же файл.
5. Если Wordstat возвращает ограничение по квоте, ноутбук просто сохраняет уже собранный прогресс и останавливается.

Здесь нет расчета остатка лимита и нет попытки угадать, сколько запросов еще осталось. Ноутбук ориентируется только на фактический ответ API.


In [6]:
# Импортируем модули для запросов к API и сохранения прогресса.
import getpass
import json
import requests
import time

WORDSTAT_API_BASE = 'https://api.wordstat.yandex.net'
TOP_REQUESTS_URL = f'{WORDSTAT_API_BASE}/v1/topRequests'


# Собираем заголовки авторизации для API.
def auth_headers(token: str) -> dict:
    return {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json; charset=utf-8',
        'Accept': 'application/json',
    }


# Финально нормализуем запрос перед отправкой в Wordstat API.
def normalize_query_api(text: str) -> str:
    text = str(text).lower().replace('ё', 'е')
    text = re.sub(r'[^a-zа-я0-9\s-]', ' ', text)
    text = text.replace('-', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Превращаем список запросов из ответа API в компактную строку.
def format_related_items(items: list[dict] | None, max_items: int = 10) -> str:
    if not isinstance(items, list):
        return ''

    parts = []

    for item in items[:max_items]:
        if not isinstance(item, dict):
            continue

        phrase = item.get('phrase') or item.get('requestPhrase') or ''
        count = item.get('count') or item.get('shows') or item.get('freq') or ''
        phrase = str(phrase).strip()

        if phrase:
            parts.append(f'{phrase} — {count}' if count != '' else phrase)

    return ' | '.join(parts)


# Проверяем, что токен рабочий.
def check_token_alive(token: str) -> tuple[int | None, dict]:
    try:
        response = requests.get(
            'https://login.yandex.ru/info',
            headers={'Authorization': f'OAuth {token}'},
            timeout=20,
        )
        try:
            return response.status_code, response.json()
        except Exception:
            return response.status_code, {'raw_text': response.text[:5000]}
    except Exception as e:
        return None, {'error': str(e)}


# Отправляем один запрос в Wordstat.
def call_wordstat_one(query: str, token: str) -> dict:
    clean_query = normalize_query_api(query)
    payload = {'phrase': clean_query}

    response = requests.post(
        TOP_REQUESTS_URL,
        headers=auth_headers(token),
        json=payload,
        timeout=30,
    )

    try:
        data = response.json()
    except Exception:
        data = None

    return {
        'status_code': response.status_code,
        'ok': response.status_code == 200,
        'json': data,
        'query': clean_query,
    }


# Извлекаем частотность и список топовых запросов из ответа API.
def extract_total_and_top_requests(response_json) -> tuple[int | None, list]:
    if isinstance(response_json, dict):
        payload = response_json
    elif isinstance(response_json, list) and response_json and isinstance(response_json[0], dict):
        payload = response_json[0]
    else:
        payload = {}

    top_requests = payload.get('topRequests') or []

    total = payload.get('totalCount')
    if total is None:
        try:
            total = sum(
                int(item.get('count', 0) or 0)
                for item in top_requests
                if isinstance(item, dict)
            )
        except Exception:
            total = None

    return total, top_requests


# Приводим таблицу к единому формату.
def normalize_wordstat_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    required_text_cols = [
        'query',
        'url',
        'source_field',
        'source_text',
        'status',
        'top_requests',
    ]

    for col in required_text_cols:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').astype(str).str.strip()

    if 'status_code' not in df.columns:
        df['status_code'] = pd.NA

    if 'total_count' not in df.columns:
        df['total_count'] = pd.NA

    df['query'] = df['query'].map(normalize_query_api)
    df['total_count'] = pd.to_numeric(df['total_count'], errors='coerce')

    return df


# Загружаем существующий файл с результатами.
def load_existing_wordstat_results(path: str | Path = 'wordstat_results.xlsx') -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    if path.suffix.lower() == '.xlsx':
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    return normalize_wordstat_columns(df)


# Сохраняем итоговый файл.
def save_wordstat_results(df: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    df = normalize_wordstat_columns(df)

    if path.suffix.lower() == '.xlsx':
        df.to_excel(path, index=False)
    else:
        df.to_csv(path, index=False, encoding='utf-8-sig')

    return path


# Объединяем старые и новые результаты, оставляя лучшую запись на query.
def merge_wordstat_results(base_df: pd.DataFrame, updates_df: pd.DataFrame | None = None) -> pd.DataFrame:
    base = normalize_wordstat_columns(base_df)

    if updates_df is None or len(updates_df) == 0:
        merged = base.copy()
    else:
        updates = normalize_wordstat_columns(updates_df)
        merged = pd.concat([base, updates], ignore_index=True)

    status_rank = {
        'done': 0,
        'pending': 1,
        'error': 2,
    }

    merged['status_rank'] = merged['status'].map(status_rank).fillna(9)

    merged = (
        merged
        .sort_values(['query', 'status_rank', 'total_count'], ascending=[True, True, False])
        .drop_duplicates(subset=['query'], keep='first')
        .drop(columns=['status_rank'], errors='ignore')
    )

    merged['total_count'] = pd.to_numeric(merged['total_count'], errors='coerce')

    merged = (
        merged
        .sort_values(['total_count', 'query'], ascending=[False, True], na_position='last')
        .reset_index(drop=True)
    )

    return merged


# Готовим очередь запросов и текущее состояние результатов.
def prepare_wordstat_queue(
    top_keys_df: pd.DataFrame,
    previous_results_path: str | Path = 'wordstat_results.xlsx',
    max_queries: int = 1000
) -> tuple[pd.DataFrame, pd.DataFrame]:
    base = normalize_wordstat_columns(top_keys_df).copy()
    base['status'] = 'pending'

    existing = load_existing_wordstat_results(previous_results_path)
    current_results = merge_wordstat_results(existing, base)

    queue = (
        current_results.loc[current_results['status'].ne('done')]
        .copy()
        .head(max_queries)
        .reset_index(drop=True)
    )

    return queue, current_results


# Обрабатываем уже подготовленную очередь, сохраняем прогресс и останавливаемся на лимите.
# Расчет и показ остатка лимита здесь убраны полностью.
def fetch_wordstat_batch(
    queue_df: pd.DataFrame,
    current_results_df: pd.DataFrame,
    token: str,
    sleep_sec: float = 1.0,
    save_every: int = 20,
    out_path: str | Path = 'wordstat_results.xlsx',
) -> pd.DataFrame:
    out_path = Path(out_path)

    queue = normalize_wordstat_columns(queue_df)
    current_results = normalize_wordstat_columns(current_results_df)

    total_queries = len(queue)

    if total_queries == 0:
        print('Новых ключей для обработки нет.')
        save_wordstat_results(current_results, out_path)
        return current_results

    print(f'Файл обновлен: {out_path.resolve()}')
    print(f'Всего строк в файле: {len(current_results)}')
    print(f'Будет обработано в этом запуске: {total_queries}')

    updates = []

    for i, row in queue.iterrows():
        query = str(row.get('query', '')).strip()
        if not query:
            continue

        result = call_wordstat_one(query, token)
        status_code = result.get('status_code')

        if status_code == 429:
            current = merge_wordstat_results(current_results, pd.DataFrame(updates))
            save_wordstat_results(current, out_path)
            print('Wordstat временно остановил обработку по квоте. Текущий прогресс сохранен.')
            return current

        total_count, top_requests = extract_total_and_top_requests(result.get('json'))

        updates.append({
            'query': query,
            'url': row.get('url', ''),
            'source_field': row.get('source_field', ''),
            'source_text': row.get('source_text', ''),
            'status': 'done' if status_code == 200 else 'error',
            'status_code': status_code,
            'total_count': total_count,
            'top_requests': format_related_items(top_requests),
        })

        current_index = i + 1

        if current_index % save_every == 0 or current_index == total_queries:
            current = merge_wordstat_results(current_results, pd.DataFrame(updates))
            save_wordstat_results(current, out_path)

            done_n = int(current['status'].eq('done').sum())
            err_n = int(current['status'].eq('error').sum())
            pending_n = int(current['status'].eq('pending').sum())

            print(
                f'{current_index}/{total_queries} | '
                f'done: {done_n} | error: {err_n} | pending: {pending_n}'
            )

        time.sleep(sleep_sec)

    final_df = merge_wordstat_results(current_results, pd.DataFrame(updates))
    save_wordstat_results(final_df, out_path)

    return final_df


## Подготовка данных к запуску

Эта ячейка читает исходный файл, фильтрует страницы, собирает запросы и формирует очередь на обработку.

In [7]:
# Укажите исходный файл и лимит новых запросов за один запуск.
INPUT_PATH = 'fitstars.csv'
MAX_QUERIES = 1000

# Читаем исходную таблицу.
df_raw = load_table(INPUT_PATH)

# Приводим таблицу к единому формату колонок.
df_norm = normalize_input_table(df_raw, CFG)

# Размечаем страницы правилами.
rule_df = build_rule_markup(df_norm, CFG)

# Отдельно сохраняем страницы, прошедшие и не прошедшие правила.
content_df = rule_df[rule_df['rule_is_article'] == 1].copy().reset_index(drop=True)
filtered_df = rule_df[rule_df['rule_is_article'] == 0].copy().reset_index(drop=True)

# Параллельно пропускаем те же страницы через текстовую модель.
scored_df, model_meta = apply_article_model(rule_df, CFG)

# Показываем краткую сводку по этапам.
print(f'Исходных строк: {len(df_raw)}')
print(f'После нормализации: {len(df_norm)}')
print(f'После базовой очистки: {len(rule_df)}')
print(f'Прошло правилами: {len(content_df)}')
print(f'Отсеяно правилами: {len(filtered_df)}')

if model_meta.get('model_loaded'):
    print(f'Модель загружена из: {model_meta["artifacts_dir"]}')
    print(f'Порог модели: {model_meta["threshold"]}')

    if 'model_is_article' in scored_df.columns:
        print(f'Прошло моделью: {int(scored_df["model_is_article"].eq(1).sum())}')

    if 'rule_vs_model' in scored_df.columns:
        print(
            f'Добавлено моделью сверх правил: '
            f'{int(scored_df["rule_vs_model"].eq("правила нет, модель да").sum())}'
        )
else:
    print(f'Модель не загружена. Ожидаются артефакты в: {model_meta["artifacts_dir"]}')

print('Распределение причин фильтрации:')
display(
    filtered_df['filter_reason']
    .value_counts(dropna=False)
    .rename_axis('filter_reason')
    .reset_index(name='count')
)

if model_meta.get('model_loaded') and 'rule_vs_model' in scored_df.columns:
    print('Сопоставление правил и модели:')
    display(make_rule_model_summary(scored_df))

    print('Примеры: правила нет, модель да')
    display(
        scored_df.loc[
            scored_df['rule_vs_model'] == 'правила нет, модель да',
            ['url', 'title', 'h1', 'meta_description', 'filter_reason', 'model_proba', 'text_all_len', 'rule_vs_model']
        ]
        .sort_values(['model_proba', 'text_all_len'], ascending=[False, False])
        .head(20)
    )

    print('Примеры: правила да, модель нет')
    display(
        scored_df.loc[
            scored_df['rule_vs_model'] == 'правила да, модель нет',
            ['url', 'title', 'h1', 'meta_description', 'filter_reason', 'model_proba', 'text_all_len', 'rule_vs_model']
        ]
        .sort_values(['model_proba', 'text_all_len'], ascending=[True, False])
        .head(20)
    )

Исходных строк: 2859
После нормализации: 2859
После базовой очистки: 2728
Прошло правилами: 1424
Отсеяно правилами: 1304
Модель загружена из: D:\IDE\project_wordstat\model_artifacts
Порог модели: 0.65
Прошло моделью: 692
Добавлено моделью сверх правил: 170
Распределение причин фильтрации:


,filter_reason,count
0,non_article_section,451
1,non_info_text,188
2,first_person_page,145
3,listing_params,131
4,site_hub_text,97
5,news_page,96
6,too_little_text,73
7,dated_page,45
8,entertainment_page,29
9,review_page,12


Сопоставление правил и модели:


,rule_vs_model,count,share
0,оба считают не статьей,1134,0.4157
1,"правила да, модель нет",902,0.3306
2,оба считают статьей,522,0.1913
3,"правила нет, модель да",170,0.0623


Примеры: правила нет, модель да


,url,title,h1,meta_description,filter_reason,model_proba,text_all_len,rule_vs_model
2186,https://fitstars.ru/blog/workout/kak-ubrat-shchyoki,"Как убрать щеки - Упражнения для похудения в щеках: польза, правильная техника",Как убрать щеки: эффективные упражнения для лица в домашних условиях,"Узнайте, как упражнения для уменьшения щек помогают подтянуть овал лица, улучшить кровообращение и добиться естественной красоты без инвазивных методов. Про...",entertainment_page,0.999140,403,"правила нет, модель да"
911,https://fitstars.ru/blog/workout/lastochka,"Ласточка упражнение - техника выполнения упражнения, польза, ч","Ласточка : техника выполнения самого изящного упражнения, польза и рекомендации","Сегодня мы изучим очень красивое и изящное упражнение, пришедшее к нам из художественной гимнастики - это упражнение ласточка. Ты узнаешь, почему его исполь...",news_page,0.998342,451,"правила нет, модель да"
297,https://fitstars.ru/blog/workout/uprazhnenie-dvorniki-yavite-miru-vash-press,"Упражнение Дворники - техника, виды, какие мышцы работают",Упражнение Дворники : явите миру ваш пресс!,Что такое упражнение Дворники ? Техника выполнения упражнения. Польза упражнения для мышц пресса.,news_page,0.998088,286,"правила нет, модель да"
1652,https://fitstars.ru/blog/workout/kak-pravilno-delat-bolgarskie-vypady,"Болгарские выпады - техника выполнения, какие мышцы работают",Как правильно делать болгарские выпады: упражнения для мышц ног и техника выполнения,"Узнай, как правильно выполнять болгарские выпады для наращивания мышечной массы и улучшения техники. В статье мы расскажем о разных вариантах упражнений и п...",first_person_page,0.997662,451,"правила нет, модель да"
1866,https://fitstars.ru/blog/workout/uprazhnenie-mostik,"Упражнение мостик : виды, техника выполнения, польза и преимущества","Упражнение мостик - виды, техника выполнения, польза и преимущества","Узнай все про упражнение мостик : в чем его польза, как правильно выполнять, как с его помощью тренировать разные группы мышц.",site_hub_text,0.996870,324,"правила нет, модель да"
1825,https://fitstars.ru/blog/workout/disk-zdorovya-osnovnye-uprazhneniya-dlya-trenirovki,Диск здоровья: упражнения для тренировки для похудения - Диск здоровья польза и функции,Диск здоровья: основные упражнения для тренировки,"Узнай все о диске здоровья: что это за тренажер, в чем его польза и какие упражнения можно на нем выполнять.",site_hub_text,0.995867,341,"правила нет, модель да"
1779,https://fitstars.ru/blog/workout/uprazhnenie-na-press-skladka-tehnika-vypolneniya,Упражнение складка на пресс - техника выполнения складки,Упражнение на пресс - складка: техника выполнения,Описание и техника упражнения складка на пресс. Преимущества упражнения. Рекомендации по выполнению складки.,news_page,0.995468,307,"правила нет, модель да"
2528,https://fitstars.ru/blog/workout/uprazhneniya-na-srednyuyu-deltu-foto-i-video,"Упражнения на среднюю дельту - базовые упражнения, продвинутые методы тренировки, техника выполнения",Упражнения на среднюю дельту. Фото и видео,Техника выполнения. Растяжка и массаж после тренировки. Базовые упражнения для средней дельты.,entertainment_page,0.994268,326,"правила нет, модель да"
415,https://fitstars.ru/blog/workout/uprazhnenie-myortvyj-zhuk-kachaem-press-doma,"Упражнение Мертвый жук для пресса - техника, польза, какие мышцы работают",Упражнение мертвый жук - качаем пресс дома,Какие мышцы задействует упражнение. Техника выполнения упражнения мертвый жук . Типичные ошибки при выполнении упражнения.,news_page,0.994105,327,"правила нет, модель да"
1635,https://fitstars.ru/blog/workout/chandra-namaskar-privetstvie-lune,"Чандра Намаскар - вечерняя йога для женщин: комплекс асан, эффект и польза для здоровья",Чандра Намаскар - приветствие Луне для пробуждения женской силы,"Узнайте все о лунном приветствии Чандра Намаскар - мягкой практике, идеальной для женщин и всех, кто хочет снять стресс, улучшить сон и обрести внутренний п...",site_hub_text,0.993818,472,"правила нет, модель да"


Примеры: правила да, модель нет


,url,title,h1,meta_description,filter_reason,model_proba,text_all_len,rule_vs_model
1924,https://fitstars.ru/blog/healthy-lifestyle/trihomoniaz-infekciya-peredavaemaya-polovym-putem,"Трихомониаз - симптомы, диагностика и методы лечения инфекции","Трихомониаз: инфекция, передаваемая половым путем","Поговорим о трихомониазе - инфекционном заболевании, передающимся половым путем. В статье рассматриваются основные симптомы трихомониаза, методы диагностики...",ok,0.002320,404,"правила да, модель нет"
80,https://fitstars.ru/mincifry,Информация для минцифры РФ,,Информация для минцифры РФ на официальном сайте FitStars.ru,ok,0.003364,125,"правила да, модель нет"
2002,https://fitstars.ru/blog/healthy-lifestyle/fitstars-pomoget-vam-ispolnit-mechtu-o-krasivom-tele-v-novom-godu,FitStars и красивая фигура - с чего начать путь к мечте в новом году,Домашние тренировки онлайн с FitStars помогут вам исполнить мечту о красивом теле в Новом году,Поздравляем настоящих и будущих пользователей платформы домашних онлайн-тренировок FitStars с Новым годом.,ok,0.003526,389,"правила да, модель нет"
611,https://fitstars.ru/blog/healthy-lifestyle/fish,С чего начать формирование здорового образа жизни?,С чего начать формирование здорового образа жизни?,,ok,0.003972,159,"правила да, модель нет"
1252,https://fitstars.ru/blog/healthy-lifestyle/weight-loss-elena,Тренировки дома онлайн с платформой Fitstars - лучшая инвестиция этого года! История Елены Асановой,Тренировки дома онлайн с платформой Fitstars - лучшая инвестиция этого года! История Елены Асановой,,ok,0.004192,270,"правила да, модель нет"
1736,https://fitstars.ru/blog/healthy-lifestyle/tireotropnyj-gormon-glavnyj-regulyator-energii-i-metabolizma,Тиреотропный гормон - регулятор функции щитовидной железы,Тиреотропный гормон - главный регулятор энергии и метаболизма,"Узнаем об основном гормоне, который регулирует работу щитовидной железы. Выясним, где он вырабатывается, какие функции выполняет и почему его уровень рекоме...",ok,0.004205,455,"правила да, модель нет"
1090,https://fitstars.ru/blog/workout/s-chego-nachat-zanyatiya-na-fitstars,Путеводитель по домашним тренировкам онлайн - Fitstars,"С чего начать занятия на FitStars, или путеводитель по домашним тренировкам онлайн",С чего начать занятия на FitStars. Путеводитель по домашним тренировкам онлайн.,ok,0.004593,297,"правила да, модель нет"
225,https://fitstars.ru/blog/healthy-lifestyle/weight-gain-sergey,Иду по жизни с FitStars. 12 лет домашних тренировок 一 история Сергея,Иду по жизни с FitStars. 12 лет домашних тренировок 一 история Сергея,,ok,0.004747,209,"правила да, модель нет"
369,https://fitstars.ru/blog/healthy-lifestyle/brosit-vyzov-projti-personalnoe-dno-i-vsyo-taki-pobedit,"Бросить вызов себе - как пройти личное дно, преодолеть трудности и выйти победителем из кризиса","Бросить вызов, пройти персональное дно и все-таки победить!","История силы и преодоления: путь через внутренний кризис к новым целям, поддержке и личной победе.",ok,0.005538,363,"правила да, модель нет"
1158,https://fitstars.ru/blog/healthy-lifestyle/prime-digest,Домашние тренировки онлайн и обучающие курсы на FitStars - дайджест наших премьер,Домашние тренировки онлайн и обучающие курсы на FitStars - дайджест наших премьер,,ok,0.006011,229,"правила да, модель нет"


Наибольшую долю составляют страницы, для которых оба подхода дают одинаковое решение: 41.57% объектов оба метода относят к не-статьям, а 19.13% — к статьям. Это говорит о наличии существенной зоны согласия между ручными эвристиками и модельной классификацией.

При этом наиболее заметное расхождение наблюдается в группе правила да, модель нет — 33.06%. Анализ примеров показывает, что в эту категорию часто попадают содержательные страницы, для которых модель выдает вероятности, близкие к выбранному порогу, но не превышающие его. Это означает, что модель в таких случаях действует более строго, чем правила, и отбрасывает часть страниц, которые эвристики считают статьями.

Группа правила нет, модель да составляет только 6.23%, что существенно меньше. Следовательно, модель относительно редко расширяет множество статей по сравнению с правилами и в основном выступает как дополнительный механизм проверки, а не как источник массового доотбора.

Таким образом, интеграция модели в проект позволяет не заменять правила, а дополнять их: выявлять зоны согласия, фиксировать спорные случаи и получать более прозрачную картину качества отбора. Практически это делает систему более интерпретируемой и создает основу для дальнейшей доработки правил и порога модели.

## Формирование итогового набора и подготовка ключей

После анализа спорных случаев формируется итоговый набор страниц.  
В него входят:
- страницы, прошедшие ручные правила;
- страницы, которые правила не отобрали, но модель классифицировала как статьи, потому что среди них оказалось много материалов, похожих на полноценные статьи. Это показывает, что правила могут отсекать часть полезных страниц. Поэтому модель в данном случае используется как способ расширить итоговую выборку и уменьшить число пропущенных статей.

Далее именно по этому объединенному набору строятся ключевые фразы и подготавливается очередь запросов в Wordstat.


In [8]:
# Формируем итоговый набор страниц после анализа сводных таблиц.
# Если модель загружена, добавляем случаи 'правила нет, модель да'.
# Если модель не загружена, используем только страницы, прошедшие правила.

if model_meta.get('model_loaded') and 'rule_vs_model' in scored_df.columns:
    final_content_df = pd.concat(
        [
            content_df.assign(article_source='rules'),
            scored_df.loc[
                scored_df['rule_vs_model'] == 'правила нет, модель да'
            ].assign(article_source='model_only')
        ],
        ignore_index=True
    ).drop_duplicates(subset=['url']).reset_index(drop=True)
else:
    final_content_df = content_df.copy()
    final_content_df['article_source'] = 'rules'

print(f'Итоговый набор статей: {len(final_content_df)}')

print('Распределение по источнику попадания в итоговый набор:')
display(
    final_content_df['article_source']
    .value_counts(dropna=False)
    .rename_axis('article_source')
    .reset_index(name='count')
)

print('Превью итогового набора:')
display(
    final_content_df[
        ['url', 'title', 'h1', 'meta_description', 'article_source']
    ].head(20)
)

# Формируем ключевые фразы уже по объединенному итоговому набору.
top_keys_df = build_candidate_queries(final_content_df, CFG)

# Подготавливаем имя итогового файла результатов.
SITE_DOMAIN = extract_main_domain(df_norm['url'])
RESULTS_PATH = f'wordstat_results_{SITE_DOMAIN}.xlsx'

# Готовим очередь новых запросов и текущее состояние итогового файла.
queue_df, results_df = prepare_wordstat_queue(
    top_keys_df=top_keys_df,
    previous_results_path=RESULTS_PATH,
    max_queries=MAX_QUERIES
)

# Сразу сохраняем общий файл, даже если часть запросов еще pending.
save_wordstat_results(results_df, RESULTS_PATH)

print(f'Итоговый файл: {Path(RESULTS_PATH).resolve()}')
print(f'Уникальных запросов в файле: {len(results_df)}')
print(f'Новых запросов в этом запуске: {len(queue_df)}')
print(f'Уже обработано: {int(results_df["status"].eq("done").sum())}')

print('Превью очереди на запуск:')
display(queue_df.head(20))

Итоговый набор статей: 1594
Распределение по источнику попадания в итоговый набор:


,article_source,count
0,rules,1424
1,model_only,170


Превью итогового набора:


,url,title,h1,meta_description,article_source
0,https://fitstars.ru/blog/workout/5-zarubezhnyh-bestsellerov-o-tom-kak-trenirovatsya-bez-boli-i-plato,"Выжимка из 5 зарубежных бестселлеров: как на самом деле тренироваться, чтобы укрепить колени, спину и тело","Что фитнес-блогеры вам не расскажут - переводим 5 зарубежных бестселлеров о том, как тренироваться без боли и плато","Собрали 15+ рабочих техник из 5 зарубежных бестселлеров для тех, кто устал от бесполезных советов. Укрепите колени, снимите боль в спине и адаптируйте нагру...",rules
1,https://fitstars.ru/recipes,Рецепты для правильного питания на каждый день от FitStars,Полезные рецепты,"Ищете рецепты для здорового и правильного питания на каждый день? FitStars предлагает множество вкусных и питательных блюд, которые помогут поддержать ваше ...",rules
2,https://fitstars.ru/blog/workout/domashnyaya-trenirovka-animal-flow-effektivnaya-fitnes-programma,Animal Flow: как через движения животных вернуть телу его природную мощь и интеллект.,"Почему все говорят про Animal Flow: практика, которая возвращает телу интеллект хищника","Разбираем 3 базовые формы Animal Flow: Зверь , Краб и Обезьяна . Узнайте, как эта практика развивает мобильность и силу, сжигая до 400 ккал за тренировку бе...",rules
3,https://fitstars.ru/blog/workout/kajtsyorfing-s-nulya-kak-bezopasno-vstat-na-dosku-i-pojmat-veter,Кайтсерфинг с нуля: как безопасно встать на доску и поймать ветер,Кайтсерфинг с нуля: как безопасно встать на доску и поймать ветер,,rules
4,https://fitstars.ru/blog/healthy-lifestyle/top-5-procedur-kotorye-bezopasny-dlya-kozhi-dazhe-letom,Летний уход за кожей: рекомендованные косметологические процедуры,"Топ-5 процедур, которые безопасны для кожи даже летом","Не все косметологические процедуры возможно делать летом. Расскажем о безопасных и результативных процедурах, подходящих для летнего сезона.",rules
5,https://fitstars.ru/blog/healthy-lifestyle/zanyatsya-sportom-chtoby-perezhit-rasstavanie-i-poluchit-figuru-mechty-istoriya-dari-firsovoj,Как с помощью домашних тренировок справиться с депрессией и похудеть,"Заняться спортом, чтобы пережить расставание и получить фигуру мечты - история Дарьи Фирсовой","История Дарьи, которая похудела на 7 кг за месяц, прокачала выносливость и подняла самооценку с помощью тренировок FitStars",rules
6,https://fitstars.ru/blog/healthy-lifestyle/treshchiny-v-ugolkah-gub-iz-za-chego-byvayut-zaedy-i-kak-ih-lechit,"Заеды в уголках рта: причины, лечение и профилактика",Трещины в уголках губ: из-за чего бывают заеды и как их лечить,Почему появляются заеды и что помогает от трещин в уголках рта. Советы врачей по лечению и профилактике.,rules
7,https://fitstars.ru/blog/healthy-lifestyle/limfodrenazhnyj-massazh-nog-vidy-tehniki-vypolneniya-i-protivopokazaniya,"Лимфодренажный массаж ног: виды, показания, противопоказания","Лимфодренажный массаж ног - виды, техники выполнения и противопоказания","Разбираем медицинский и косметический лимфодренажный массаж. Когда нужен врач, а когда достаточно салона красоты. Противопоказания и техники выполнения.",rules
8,https://fitstars.ru/blog/workout/hiit-light-ot-lizy-prokudinoj-kogda-intervalnye-trenirovki-rabotayut-na-metabolizm-ne-na-pereutomlenie,HIIT LIGHT: как интервальные тренировки помогают худеть новичкам без изнурения,"HIIT LIGHT от Лизы Прокудиной - когда интервальные тренировки работают на метаболизм, а не на переутомление","Разбираем программу HIIT LIGHT от Елизаветы Прокудиной. Это 20 тренировок по 25 минут для плавного старта. Узнайте, как сжигать до 200 ккал, развивать вынос...",rules
9,https://fitstars.ru/blog/nutrition/kumkvat-malenkij-frukt-s-bolshoj-polzoj,https://fitstars.ru/blog/nutrition/kumkvat-malenkij-frukt-s-bolshoj-polzoj,Кумкват - маленький фрукт с большой пользой,,rules


Итоговый файл: D:\IDE\project_wordstat\wordstat_results_fitstars_ru.xlsx
Уникальных запросов в файле: 1547
Новых запросов в этом запуске: 1000
Уже обработано: 0
Превью очереди на запуск:


,query,url,source_field,source_text,status,top_requests,status_code,total_count
0,barre фитнес,https://fitstars.ru/blog/workout/barre-fitnes-chto-eto-takoe-polza-i-kak-nachat-zanimatsya-doma,h1,"Barre фитнес: что это такое, польза и как начать заниматься дома?",pending,,NaN,NaN
1,body combat,https://fitstars.ru/blog/workout/body-combat,h1,Body Combat: задайте себе физическую и психологическую встряску!,pending,,NaN,NaN
2,body sculpt,https://fitstars.ru/blog/healthy-lifestyle/body-sculpt-stroim-telo,h1,Body Sculpt. Строим тело,pending,,NaN,NaN
3,chloe ting,https://fitstars.ru/blog/healthy-lifestyle/chloe-ting-kak-postroit-telo-mechty,h1,Chloe Ting: как повторить ее успех и построить тело мечты?,pending,,NaN,NaN
4,ems тренировки,https://fitstars.ru/blog/workout/ems-trenirovki-kak-za-25-minut-poluchit-effekt-2-chasovogo-zanyatiya,h1,"EMS-тренировки - упражнения для мышц, как за 25 минут получить эффект",pending,,NaN,NaN
5,fullbody революция,https://fitstars.ru/blog/workout/fullbody-revolyuciya-kak-prokachat-vse-telo-za-odnu-trenirovku,h1,Fullbody-революция: как прокачать все тело за одну тренировку,pending,,NaN,NaN
6,hiit light,https://fitstars.ru/blog/workout/hiit-light-ot-lizy-prokudinoj-kogda-intervalnye-trenirovki-rabotayut-na-metabolizm-ne-na-pereutomlenie,h1,"HIIT LIGHT от Лизы Прокудиной - когда интервальные тренировки работают на метаболизм, а не на переутомление",pending,,NaN,NaN
7,hiit тренировки для похудения и роста мышц,https://fitstars.ru/blog/workout/slimming-workout,h1,HIIT тренировки для похудения и роста мышц - в чем секрет их популярности,pending,,NaN,NaN
8,hustle culture и зож,https://fitstars.ru/blog/healthy-lifestyle/hustle-culture-i-zozh-kak-obresti-balans,h1,Hustle culture и ЗОЖ - как обрести баланс,pending,,NaN,NaN
9,lpg массаж,https://fitstars.ru/blog/healthy-lifestyle/lpg-massazh-korrekciya-figury-bez-operacii,h1,LPG-массаж: коррекция фигуры без операции,pending,,NaN,NaN


## Проверка токена

Эта ячейка проверяет OAuth-токен Wordstat перед запуском запросов.

Что именно происходит:
1. Сначала токен берется из переменной окружения `WORDSTAT_TOKEN`.
2. Если переменная пустая, ноутбук просит ввести токен вручную.
3. Потом выполняется простой запрос проверки токена.
4. В ответ выводится статус и служебная информация, чтобы было видно, что токен живой.

Эта проверка подтверждает доступ, но не используется для расчета остатка дневного лимита.

### Для работы необходим токен wordstat

In [9]:
# Проверяем токен перед отправкой запросов в API.
WORDSTAT_TOKEN = os.getenv('WORDSTAT_TOKEN', '').strip()
if not WORDSTAT_TOKEN:
    # Если токен не задан в окружении, просим ввести его вручную.
    WORDSTAT_TOKEN = getpass.getpass('Введите OAuth token Wordstat: ').strip()

if not WORDSTAT_TOKEN:
    raise ValueError('Токен не введен.')

# Проверяем, что токен живой и отвечает.
status_code, token_info = check_token_alive(WORDSTAT_TOKEN)
print('token status:', status_code)
print(token_info)



token status: 200
{'id': '308652763', 'login': 'lenalenalenaaaaa', 'client_id': '5ab61ddce12f4c4eb60d09c075ba5980', 'default_email': 'lenalenalenaaaaa@yandex.ru', 'emails': ['lenalenalenaaaaa@yandex.ru'], 'psuid': '1.AA_Vrg.YyG-xGjm0NWKyM24hTaGMA.Xs0_DKyOie6ILbz_jXWU5w'}


In [10]:
# Запускаем обработку очереди и обновляем один итоговый файл по ходу работы.
results_df = fetch_wordstat_batch(
    queue_df=queue_df,
    current_results_df=results_df,
    token=WORDSTAT_TOKEN,
    sleep_sec=0.1,
    save_every=20,
    out_path=RESULTS_PATH,
)

print(f'Готово. Файл сохранен в: {Path(RESULTS_PATH).resolve()}')

# Показываем, где лежит итоговый файл и сколько строк в нем сейчас.
print(f'Всего строк в файле: {len(results_df)}')
display(results_df.head(30))


# Показываем обновленное распределение статусов после расчета частотностей.
final_status_summary_df = (
    results_df['status']
    .fillna('pending')
    .value_counts(dropna=False)
    .rename_axis('status')
    .reset_index(name='count')
)

print('Статусы после обновления файла:')
display(final_status_summary_df)

done_count = int(results_df['status'].fillna('').eq('done').sum())
pending_count = int(results_df['status'].fillna('').isin(['pending', 'error']).sum())

print(f'Готовых ключей: {done_count}')
print(f'Незавершенных ключей: {pending_count}')


Файл обновлен: D:\IDE\project_wordstat\wordstat_results_fitstars_ru.xlsx
Всего строк в файле: 1547
Будет обработано в этом запуске: 1000
20/1000 | done: 20 | error: 0 | pending: 1527
40/1000 | done: 40 | error: 0 | pending: 1507
60/1000 | done: 60 | error: 0 | pending: 1487
80/1000 | done: 80 | error: 0 | pending: 1467
100/1000 | done: 100 | error: 0 | pending: 1447
120/1000 | done: 120 | error: 0 | pending: 1427
140/1000 | done: 140 | error: 0 | pending: 1407
160/1000 | done: 160 | error: 0 | pending: 1387
180/1000 | done: 180 | error: 0 | pending: 1367
200/1000 | done: 200 | error: 0 | pending: 1347
220/1000 | done: 220 | error: 0 | pending: 1327
240/1000 | done: 240 | error: 0 | pending: 1307
260/1000 | done: 260 | error: 0 | pending: 1287
280/1000 | done: 280 | error: 0 | pending: 1267
300/1000 | done: 300 | error: 0 | pending: 1247
320/1000 | done: 320 | error: 0 | pending: 1227
340/1000 | done: 340 | error: 0 | pending: 1207
360/1000 | done: 360 | error: 0 | pending: 1187
380/100

,query,url,source_field,source_text,status,top_requests,status_code,total_count
0,матча,https://fitstars.ru/blog/nutrition/matcha-zelyonyj-eliksir-zdorovya-i-energii,h1,Матча - зеленый эликсир здоровья и энергии,done,матч — 47407616 | матчи кхл — 8266654 | матчи чемпионатов — 3992043 | россия матчи — 3833204 | чемпионат россии матчи — 3136358 | матч тв — 2752810 | распис...,200,47424570.0
1,как и,https://fitstars.ru/blog/healthy-lifestyle/we-start-to-lose-weight,h1,Как и с чего начинать похудение - пошаговая инструкция,done,как и — 36673881 | как и в — 14197909 | как и на — 9217160 | как и им — 6347791 | как и с — 6047303 | как и чем — 5877300 | что и как — 5877154 | и как ему ...,200,36673881.0
2,легко,https://fitstars.ru/blog/nutrition/otkazatsya-ot-molochki-legko-poshagovaya-instrukciya,h1,Отказаться от молочки? Легко! Пошаговая инструкция,done,легкие — 10070761 | легкие рисунки — 731258 | какие легкие — 566010 | легко 2 — 547832 | легко 1 — 527490 | легкие 3 — 460989 | суть легка — 410542 | самый ...,200,10070706.0
3,горы,https://fitstars.ru/blog/workout/gory-ne-ekstrim-nauka-kak-vybrat-svoj-tip-pohoda-podgotovit-telo-i-sobrat-ryukzak-novichku,h1,"Горы - не экстрим, а наука: как выбрать свой тип похода, подготовить тело и собрать рюкзак новичку",done,горе — 8012926 | горы — 7844292 | горе гора — 7842965 | горе 2 — 599360 | горе 1 — 572367 | 1 гор — 551786 | горе 3 — 494295 | 3 гор — 474427 | гора 1 2 — 4...,200,7844292.0
4,ванна,https://fitstars.ru/blog/healthy-lifestyle/vanna-polza-i-vred,h1,Ванна - польза и вред: что нужно знать спортсменам,done,ванна — 6126884 | ванная — 3541017 | ванныя — 3540815 | комната ванна — 686991 | ванная комната — 679838 | ванныя комната — 679828 | ванна купить — 377631 |...,200,6126884.0
5,зал,https://fitstars.ru/blog/workout/zal-eto-baza-fitstars-apgrejd-chem-onlajn-trenirovki-polezny-tem-kto-uzhe-v-teme,h1,"Зал - это база, а FitStars - апгрейд: чем онлайн-тренировки полезны тем, кто уже в теме",done,зал — 4338029 | залы — 4321559 | тренажерный зал — 384064 | тренажерные залы — 384056 | концертный зал — 368801 | концертные залы — 368796 | бизнес зал — 25...,200,4338029.0
6,зрение,https://fitstars.ru/blog/healthy-lifestyle/zrenie-kak-sohranit-i-uluchshit-v-lyubom-vozraste,h1,Зрение: как сохранить и улучшить в любом возрасте - рассказывают офтальмологи,done,зрение — 3298051 | точка зрения — 932680 | какое зрение — 385843 | коррекция зрения — 261641 | зрение 1 — 251538 | 2 зрение — 243722 | можно зрение — 202498...,200,3298051.0
7,на море,https://fitstars.ru/blog/healthy-lifestyle/na-more-s-ivannoj-idush,h1,"На море с Иванной Идуш: стройнеть, заряжаться, оздоравливаться",done,на море — 3109802 | на море 2 — 377436 | на море 1 — 372977 | море на 3 — 341196 | на берегу моря — 337312 | моря на карте — 299629 | на каком море — 292073...,200,3109802.0
8,клещи,https://fitstars.ru/blog/healthy-lifestyle/kleshchi-i-rech-ne-ob-instrumente,h1,Клещи. И речь не об инструменте,done,клещами клещи — 2790324 | клещ — 2789615 | прививка от клеща — 270725 | от клещей для собак — 263432 | таблетки от клещей — 152104 | укус клещами — 129025 |...,200,2795773.0
9,июль,https://fitstars.ru/blog/healthy-lifestyle/iyul-kajfuj-uspejte-kupit-podpisku-fitstars-po-akcii,h1,Июль - кайфуй! : успейте купить подписку FitStars по акции,done,июль — 2175343 | январь июль — 226847 | 1 июля — 219020 | г июля — 212537 | июль 2026 — 211918 | июль какой — 207969 | июнь июль — 181556 | температура июля...,200,2175343.0


Статусы после обновления файла:


,status,count
0,done,779
1,pending,768


Готовых ключей: 779
Незавершенных ключей: 768


**Вывод:** Полностью универсальными правилами нельзя идеально определить корректный ключ для любой статьи, потому что сайты отличаются по структуре, стилю заголовков и количеству шумовых формулировок. Поэтому разработанное решение не заменяет специалиста, а помогает сократить объем ручной работы.

В проект дополнительно встроена модель классификации, которая определяет, является страница статьей или нет. Она дополняет правила, помогает находить спорные случаи и уменьшает риск пропуска полезных материалов.

В результате пользователь получает автоматически подготовленный набор ключевых фраз, очищенный от части нерелевантных страниц и типичных шумовых формулировок. Система также учитывает частотность запросов через Wordstat и сохраняет прогресс обработки.

Таким образом, цель проекта достигнута: создан инструмент, который автоматизирует подбор ключевых фраз для информационных статей, ускоряет работу с большими массивами страниц и делает отбор более полным за счет совместного использования правил, модели и Wordstat.